# Aula 1.3 Explorando o Databricks
## Bibliotecas e Ferramentas Pré-instaladas no Ambiente

Uma das grandes vantagens de utilizar o **Databricks Runtime para Machine Learning** é que o cluster já vem configurado com as principais ferramentas do ecossistema de Ciência de Dados

Abaixo estão alguns exemplos do que já está disponível pronto para uso:

* **Scikit-Learn:** Para preprocessamento clássico, métricas e modelos em nós únicos.
* **Apache Spark / PySpark:** O motor principal para manipulação de grandes volumes de dados (camadas Bronze, Silver e Gold).
* **Delta Lake:** Camada de armazenamento ACID que garante a confiabilidade das tabelas de e-commerce.
* **MLflow:** Integrado de forma nativa para fazer o tracking automático de parâmetros e registrar o modelo final no Unity Catalog

In [0]:
%pip list

In [0]:
import sklearn
import mlflow
import pyspark

print("scikit-learn:", sklearn.__version__)
print("mlflow:", mlflow.__version__)
print("pyspark:", pyspark.__version__)

abaixo é um teste meu para acompanhar a aula da alura

In [0]:
%sql
select * from samples.accuweather.forecast_daily_calendar_imperial limit 100

## Gerenciamento e Instalação de Pacotes Adicionais (`%pip`)

Embora o ambiente nativo do Databricks ML já traga a grande maioria das ferramentas necessárias, existem cenários onde precisamos de bibliotecas externas específicas ou de uma versão diferente de algum pacote. 

Para isso, o Databricks oferece suporte aos **comandos de escopo do notebook** usando o gerenciador de pacotes `pip`. 

* Usamos o comando `%pip install` precedido pelo caractere `%` (magic command).

In [0]:
%pip install torch torchvision torchaudio

In [0]:
import torch
print("torch:", torch.__version__)

In [0]:
%pip install xgboost

In [0]:
%restart_python

In [0]:
#%pip install xgboost
import xgboost
print("xgboost:", xgboost.__version__)

## Monitoramento Pragmático de Hardware via Python

* `os` e `platform`: Para mapear a quantidade de núcleos de processamento (vCPUs) e a arquitetura do sistema operacional subjacente.
* `psutil`: Biblioteca poderosa para monitorar o uso de recursos do sistema, como consumo detalhado de memória RAM e processos ativos.
* `shutil`: Utilizada para verificar o espaço total, ocupado e disponível nos discos rígidos e sistemas de arquivos montados (`/`).

In [0]:
import os
import psutil
import platform

print(f"CPU Cores: {os.cpu_count()}")
print(f"RAM Total: {round(psutil.virtual_memory().total / (1024**3), 2)} GB")
print(f"RAM Available: {round(psutil.virtual_memory().available / (1024**3), 2)} GB")
print(f"OS: {platform.platform()}")

In [0]:
import shutil

total, used, free = shutil.disk_usage("/")

print(f"Disk Total: {round(total / (1024**3), 2)} GB")
print(f"Disk Used: {round(used / (1024**3), 2)} GB")
print(f"Disk Free: {round(free / (1024**3), 2)} GB")

# Integração ao GitHub
O Databricks possui uma integração nativa com provedores de Git (como GitHub, GitLab e Bitbucket) chamada **Databricks Repos** (ou *Git Folders*). Essa funcionalidade elimina a necessidade de exportar arquivos manualmente para sincronizar o código com o seu repositório.

# Databricks notebook, consultado o Unity Catalog

Neste ambiente, utilizamos **Notebooks interativos**, que combinam documentação e código em um único lugar. Uma das grandes vantagens do Databricks é a sua natureza poliglota:

* **Markdown e Python:** São as linguagens padrão para documentação, manipulação de dados e Machine Learning.
* **Ambiente Poliglota (%):** Através das *magic commands*, podemos alternar a linguagem de uma célula específica instantaneamente.
    * `%sql`: Para rodar consultas SQL diretas nas tabelas do Lakehouse.
    * `%r`: Para análises estatísticas usando a linguagem R *(Disponível nativamente em ambientes corporativos/Enterprise, onde há suporte a múltiplos nós e clusters customizados. Na versão gratuita/Community, o foco principal fica em Python e SQL devido às limitações de recursos do cluster único).*

In [0]:
%sql
select * from samples.accuweather.forecast_daily_calendar_imperial limit 100

In [0]:
# Fazendo a mesma leitura, mas usando a sintaxe Python/PySpark
df_previsao = spark.read.table("samples.accuweather.forecast_daily_calendar_imperial")

# Exibindo os dados com o visualizador interativo do Databricks
display(df_previsao.limit(100))

A Função `display()` — Visualização Interativa
É uma ferramenta exclusiva do Databricks que substitui o tradicional `.show()` do PySpark.
* **O que faz:** Renderiza os dados em uma tabela gráfica interativa e elegante.

O Método `summary()` — Diagnóstico Estatístico
É um método nativo do PySpark DataFrame para levantamento estatístico descritivo.
* **O que faz:** Calcula automaticamente as métricas centrais e de dispersão para todas as colunas numéricas da tabela.
* **Métricas inclusas:** Contagem de registros válidos (`count`), média (`mean`), desvio padrão (`stddev`), valores mínimos e máximos (`min`/`max`), além dos quartis calculados (25%, 50% e 75%).

In [0]:
# sumario com o spark
display(df_previsao.summary())

## prompt para Genie Code

Usando a tabela samples.accuweather.forecast_daily_calendar_imperial, escreva um código em Python usando a biblioteca seaborn para criar um gráfico de barras.
O gráfico deve mostrar a média da temperatura máxima (procure pela coluna de temperatura máxima, geralmente max_temperature ou similar) agrupada pelo dia da semana (ou mês, caso esteja disponível).
Por favor, ordene as barras cronologicamente, adicione um título descritivo em português, rotule os eixos X e Y de forma clara e use uma paleta de cores degradê azul ou laranja.


## Leitura Direta de Arquivos nos Volumes com Spark SQL

No Unity Catalog, utilizamentos **Volumes** para armazenar arquivos brutos (como CSVs, JSONs e logs) antes de transformá-los em tabelas Delta. 

Através do Spark SQL, podemos utilizar a função `read_files()`

Vamos salvar a tabela como DELTA TABLE
- Transações ACID
- Versionamento e Time Travel
- Suporte a MERGE (UPSERT) e evolução de esquema

In [0]:
import pyspark

# este caminho aqui embaixo estava vazio no código da aula, eu criei um volume em catalogs, subi o arquivo e colei aqui o caminho completo

caminho_csv = "/Volumes/workspace/default/workspace/base_dim_produtos.csv"
tabela_destino = "workspace.default.base_dim_produtos"

print("Iniciando a ingestão com Esquema Explícito e Tipagem Estrita...")

# Executando a criação com a definição exata das colunas (DDL Schema)
spark.sql(f"""
    CREATE OR REPLACE TABLE {tabela_destino}
    USING DELTA
    AS 
    SELECT * FROM read_files(
      '{caminho_csv}',
      format => 'csv',
      header => 'true',
      schema => '
        id_produto STRING, 
        nome_produto STRING, 
        categoria STRING, 
        preco_reais DECIMAL(18,2)
      '
    )
""")

print(f"Sucesso! Tabela Delta gravada de forma otimizada em: {tabela_destino}")

# Verificando se a tipagem foi aplicada corretamente no catálogo
df_schema = spark.sql(f"DESCRIBE TABLE {tabela_destino}")
display(df_schema[["col_name", "data_type"]])

## Simulação de Desastre e Recuperação com Delta Time Travel

Um dos recursos mais poderosos do Delta Lake é o **Log de Transações (Delta Transaction Log)**. Cada operação de escrita, alteração ou exclusão gera uma nova versão da tabela (Versão 0, Versão 1, Versão 2...).

Para demonstrar a resiliência do Lakehouse, vamos simular um erro crítico de produção:
1. Vamos ler a tabela de produtos original do Volume.
2. **O Erro:** Um analista roda um pipeline que acidentalmente remove uma coluna inteira (ex: `preco_reais`) ou sobrescreve a tabela com dados incompletos.
3. **A Solução:** Utilizaremos o **Time Travel** para consultar o histórico de modificações, identificar a versão correta do passado e restaurar a tabela ao seu estado original em segundos, sem precisar reprocessar arquivos ou buscar backups externos.

In [0]:
import pyspark
tabela_produtos = "workspace.default.base_dim_produtos"
# ---- PASSO 2: O ERRO! Sobrescrevendo a tabela sem a coluna preco_reais (Cria a Versão 1) ----
print("Simulando erro humano... Pipeline rodou dropando a coluna 'preco_reais'!")
spark.sql(f"""
    CREATE OR REPLACE TABLE {tabela_produtos}
    USING DELTA
    AS 
    SELECT
    id_produto,
    nome_produto,
    categoria,
    0 as preco_reais
    FROM
    {tabela_produtos}
""")

print("\n❌ Desastre concluído. Vamos ver como a tabela ficou na produção:")
display(spark.sql(f"SELECT * FROM {tabela_produtos}"))

## Investigando o Histórico da Tabela (`DESCRIBE HISTORY`)

Fomos avisados de que o modelo preditivo quebrou porque a coluna `preco_reais` sumiu. Antes de restaurar, precisamos olhar o livro de registros da tabela Delta para entender o que aconteceu e descobrir qual era a versão saudável.

In [0]:
# Investigando o histórico de modificações da tabela Delta
df_historico = spark.sql(f"DESCRIBE HISTORY {tabela_produtos}")

# Exibindo as colunas mais importantes do log de auditoria
display(df_historico[["version", "timestamp", "userId", "operation", "operationParameters"]])

# aqui deram versão 0 e 1, depois eu mudei o código acima para colocar preco = 100  ( 100 as preco_reais ) e isto gerou a vesrsão 2

### Executando o Time Travel e Restaurando os Dados

Agora que identificamos que a **Versão 0** continha os dados corretos com a coluna de preços, temos duas formas de resolver:
1. **Consulta Rápida:** Ler a tabela apontando para a versão antiga (`VERSION AS OF`).
2. **Restauração Definitiva:** Usar o comando `RESTORE` para forçar a tabela atual a voltar no tempo.

In [0]:
print("🔄 Executando o RESTORE para voltar a tabela para a Versão 0...")

# O comando RESTORE faz a tabela voltar exatamente ao estado da versão escolhida
spark.sql(f"RESTORE TABLE {tabela_produtos} TO VERSION AS OF 0")

print("\n✅ Tabela restaurada com sucesso! Vamos validar se a coluna 'preco_reais' voltou:")
df_recuperado = spark.sql(f"SELECT * FROM {tabela_produtos}")
display(df_recuperado)

## Fazendo com SQL

In [0]:
%sql
RESTORE TABLE workspace.default.base_dim_produtos TO VERSION AS OF 0


In [0]:
%sql
select * from workspace.default.base_dim_produtos